# ContinuityGuard · RASCALS
## 02 — Correctness and inference benchmark

Run after 01. These tests exercise causal features, scenario reproducibility, deduplication, bounded storage, incident timing, model integrity, authentication, and the live sandbox. The benchmark measures inference processing, not source-to-browser delay or production capacity.

In [1]:
from pathlib import Path
import os, sys, platform, json
ROOT = Path.cwd()
if not (ROOT / 'continuityguard').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'continuityguard').is_dir(), 'Open this notebook from the project notebooks folder'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('Team: RASCALS')
print('Architecture:', platform.machine())
print('Python:', platform.python_version())
print('Project:', ROOT)
assert platform.machine() == 's390x', 'This submission notebook is intended to run on LinuxONE'


Team: RASCALS
Architecture: s390x
Python: 3.12.15
Project: /home/ibm-user/shared/continuityguard


In [2]:
import subprocess
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'Correctness checks failed; fix before submission'



/opt/venv/lib64/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
test_actual_chunked_body_limit_is_enforced (test_system.ApiTests.test_actual_chunked_body_limit_is_enforced) ... ok
test_consumer_failure_is_visible_and_does_not_deadlock_queue (test_system.ApiTests.test_consumer_failure_is_visible_and_does_not_deadlock_queue) ... ok
test_control_actions_do_not_accept_shell_commands (test_system.ApiTests.test_control_actions_do_not_accept_shell_commands) ... ok
test_event_journal_survives_reopen (test_system.ApiTests.test_event_journal_survives_reopen) ... ok
test_excessive_batch_is_rejected (test_system.ApiTests.test_excessive_batch_is_rejected) ... ok
test_external_events_cannot_impersonate_sandbox_provenance (test_system.ApiTests.test_external_events_cannot_impersonate_sandbox_provenance) ... ok
test_fracti

In [3]:
from continuityguard.benchmark import main
main(ROOT / 'artifacts')

{"offered_events_per_second": 10, "accepted_events": 80, "rejected_events": 0, "duplicates": 0, "measurement_seconds": 8.0, "achieved_events_per_second": 10.0, "batch_size": 4, "p50_batch_processing_ms": 8.328353998876992, "p95_batch_processing_ms": 9.904860449751151, "p99_batch_processing_ms": 20.62647608763653, "process_cpu_seconds": 0.16, "process_rss_bytes": 181493760}


{"offered_events_per_second": 50, "accepted_events": 400, "rejected_events": 0, "duplicates": 0, "measurement_seconds": 8.0, "achieved_events_per_second": 50.0, "batch_size": 4, "p50_batch_processing_ms": 7.696331498664222, "p95_batch_processing_ms": 9.180583100351214, "p99_batch_processing_ms": 13.268971170728044, "process_cpu_seconds": 0.68, "process_rss_bytes": 184393728}


{"offered_events_per_second": 100, "accepted_events": 800, "rejected_events": 0, "duplicates": 0, "measurement_seconds": 8.0, "achieved_events_per_second": 100.0, "batch_size": 4, "p50_batch_processing_ms": 7.17701350004063, "p95_batch_processing_ms": 8.68683645094279, "p99_batch_processing_ms": 11.20304085045062, "process_cpu_seconds": 1.25, "process_rss_bytes": 185262080}


{'architecture': 's390x',
 'logical_cpus': 3,
 'duration_per_trial_seconds': 8,
 'scope': 'Paced in-process engine with SQLite commits on the assigned LinuxONE VM. Source telemetry is synthetic service-window events; this is not an HTTP or source-to-browser load test. GenAI is off.',
 'boundary': 'Feature extraction, ML scoring, alert policy and journal commit; measured for batches of 4 events.',
 'queue_scope': 'Direct engine: ingestion queue is bypassed in this benchmark. Queue backpressure is checked separately by correctness tests.',
 'trials': [{'offered_events_per_second': 10,
   'accepted_events': 80,
   'rejected_events': 0,
   'duplicates': 0,
   'measurement_seconds': 8.0,
   'achieved_events_per_second': 10.0,
   'batch_size': 4,
   'p50_batch_processing_ms': 8.328353998876992,
   'p95_batch_processing_ms': 9.904860449751151,
   'p99_batch_processing_ms': 20.62647608763653,
   'process_cpu_seconds': 0.16,
   'process_rss_bytes': 181493760},
  {'offered_events_per_second': 50

In [4]:
display(json.loads((ROOT / 'artifacts/runtime-benchmark.json').read_text()))

{'architecture': 's390x',
 'logical_cpus': 3,
 'duration_per_trial_seconds': 8,
 'scope': 'Paced in-process engine with SQLite commits on the assigned LinuxONE VM. Source telemetry is synthetic service-window events; this is not an HTTP or source-to-browser load test. GenAI is off.',
 'boundary': 'Feature extraction, ML scoring, alert policy and journal commit; measured for batches of 4 events.',
 'queue_scope': 'Direct engine: ingestion queue is bypassed in this benchmark. Queue backpressure is checked separately by correctness tests.',
 'trials': [{'offered_events_per_second': 10,
   'accepted_events': 80,
   'rejected_events': 0,
   'duplicates': 0,
   'measurement_seconds': 8.0,
   'achieved_events_per_second': 10.0,
   'batch_size': 4,
   'p50_batch_processing_ms': 8.328353998876992,
   'p95_batch_processing_ms': 9.904860449751151,
   'p99_batch_processing_ms': 20.62647608763653,
   'process_cpu_seconds': 0.16,
   'process_rss_bytes': 181493760},
  {'offered_events_per_second': 50

### Interpretation
The reported throughput is a timed in-process benchmark on the assigned VM, using synthetic inputs. It is not a load test of the full HTTP ingestion endpoint or a platform-wide performance claim. CPU quota, offered volume, dropped/rejected counts and measurement boundaries are reported explicitly.